In [1]:
import pandas as pd
import requests 
import os
import pydub

In [33]:
!pip install ipywidgets

Leer CSV y hacer carpetas por especie

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/anto-rom/Xeno_Canto_Project/refs/heads/main/data/processed/df_final.csv"
df = pd.read_csv(url)

# Muestra columnas
print("Columnas en el CSV:")
print(df.columns.tolist())

pd.set_option('display.max_colwidth', None)


# Muestra primeras filas para entender qué contiene
df.head(1)

Columnas en el CSV:
['scientificName', 'references', 'vernacularName', 'country', 'description']


,scientificName,references,vernacularName,country,description
0,acrocephalus arundinaceus,https://data.biodiversitydata.nl/xeno-canto/observation/XC730881,Great Reed Warbler,Sweden,"Acrocephalus arundinaceus, commonly known as the great reed warbler, is a large passerine bird primarily found across Europe and western Asia. It inhabits dense reed beds in freshwater wetlands, often near lakes, slow-flowing rivers, and marshes. The species is insectivorous, feeding mainly on large insects and spiders, occasionally consuming small vertebrates. Behaviorally, it is known for its loud and complex song, which plays a role in territorial defense and mate attraction. The great reed warbler is migratory, wintering in sub-Saharan Africa. Its population is currently stable, and the species is classified as Least Concern by the IUCN, although habitat loss poses localized threats."


In [12]:
import tensorflow as tf
import tensorflow_hub as hub

# Forzar a TF Hub a que use una descarga fresca si falla la anterior
os.environ["TFHUB_DOWNLOAD_PROGRESS"] = "1"

try:
    print("🔄 Descargando Google Perch (esto puede tardar unos minutos)...")
    
    # Opción A: Usar la URL de Kaggle (que es la sucesora oficial de tfhub.dev)
    model_url = "https://www.kaggle.com/models/google/bird-vocalization-classifier/tensorFlow2/bird-vocalization-classifier/1"
    
    # Cargamos el modelo
    model = hub.load(model_url)
    perch = model.signatures['serving_default']
    
    print("✅ Perch cargado correctamente")
except Exception as e:
    print(f"❌ Falló la carga desde Kaggle. Intentando URL alternativa...")
    # Opción B: URL directa de almacenamiento si la anterior falla
    try:
        model = hub.load("https://tfhub.dev/google/perch/1")
        perch = model.signatures['serving_default']
        print("✅ Perch cargado correctamente (URL alternativa)")
    except Exception as e2:
        raise RuntimeError(f"No se pudo cargar el modelo de ninguna forma: {e2}")

🔄 Descargando Google Perch (esto puede tardar unos minutos)...
Downloaded https://www.kaggle.com/models/google/bird-vocalization-classifier/tensorFlow2/bird-vocalization-classifier/1, Total size: 96.30MB

✅ Perch cargado correctamente


Descargar y converversion a embeddings de google perch 

In [ ]:
import os
import tempfile
import requests
import pandas as pd
import numpy as np
from pydub import AudioSegment
import tensorflow as tf
import warnings
from audioclass import Perch
import time

warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

CSV_URL = "https://raw.githubusercontent.com/anto-rom/Xeno_Canto_Project/refs/heads/main/data/processed/df_final.csv"
EMB_DIR = "embeddings_perch"
SAMPLE_RATE = 32000
MAX_AUDIO_SEC = 5
MIN_AUDIO_SEC = 1.0
BATCH_SIZE = 5000

os.makedirs(EMB_DIR, exist_ok=True)

# ======================================
# FUNCIONES AUXILIARES
# ======================================
def extract_xc_id(ref_url):
    return ref_url.rstrip("/").split("/")[-1].replace("XC", "")

def download_audio(xc_id):
    url = f"https://xeno-canto.org/{xc_id}/download"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        raise ValueError(f"Error al descargar XC{xc_id}")
    with tempfile.NamedTemporaryFile(suffix=".audio") as tmp:
        tmp.write(r.content)
        tmp.flush()
        audio = AudioSegment.from_file(tmp.name)
    return audio

def preprocess_audio_fixed(audio):
    audio = audio.set_channels(1).set_frame_rate(SAMPLE_RATE)
    audio = audio[:MAX_AUDIO_SEC * 1000]
    samples = np.array(audio.get_array_of_samples()).astype(np.float32)
    samples /= 32768.0
    expected_len = SAMPLE_RATE * MAX_AUDIO_SEC
    if len(samples) < expected_len:
        samples = np.pad(samples, (0, expected_len - len(samples)))
    else:
        samples = samples[:expected_len]
    return samples

# ======================================
# CARGAR CSV
# ======================================
df = pd.read_csv(CSV_URL)
total = len(df)
print(f"✅ {total} registros cargados")

# ======================================
# CARGAR PERCH
# ======================================
print("⚡ Inicializando Perch...")
perch = Perch.load_model("bird-vocalization-classifier")
print("✅ Perch cargado")

# ======================================
# LOOP PRINCIPAL CON BARRA PROGRESO POR PRINT
# ======================================
start_time = time.time()
processed_count = 0

for idx, row in enumerate(df.itertuples(index=False), 1):
    species = row.scientificName
    xc_id = extract_xc_id(row.references)
    species_dir = os.path.join(EMB_DIR, species)
    os.makedirs(species_dir, exist_ok=True)
    out_file = os.path.join(species_dir, f"XC{xc_id}.npz")

    if os.path.exists(out_file):
        continue

    try:
        audio = download_audio(xc_id)
        if len(audio) < MIN_AUDIO_SEC * 1000:
            continue
        samples = preprocess_audio_fixed(audio)
        output = perch.process_array(samples)
        embedding = output.features[0]

        np.savez_compressed(
            out_file,
            embedding=embedding.astype(np.float16),
            species=species,
            xc_id=f"XC{xc_id}"
        )
        processed_count += 1

    except Exception as e:
        continue

    # Print cada 100 audios
    if idx % 100 == 0 or idx == total:
        pct = idx / total * 100
        elapsed = time.time() - start_time
        print(f"{pct:.2f}% completado | audios procesados: {idx} | tiempo transcurrido: {elapsed/60:.2f} min")

# Limpieza final
tf.keras.backend.clear_session()
print("✅ Todos los embeddings generados")


Conocer cuantas carpetas de especies hay y cuantos embeddings en cada una

In [23]:
#para saber cuantas carpetas hay y cuantos embeddings en cada una 

import os
import pandas as pd

EMB_DIR = "embeddings_perch"

data = []

for species in os.listdir(EMB_DIR):
    species_path = os.path.join(EMB_DIR, species)
    if not os.path.isdir(species_path):
        continue

    num_emb = len([
        f for f in os.listdir(species_path)
        if f.endswith(".npz")
    ])

    data.append({
        "species": species,
        "num_embeddings": num_emb
    })

df_counts = pd.DataFrame(data).sort_values("num_embeddings", ascending=False)

print(f"🐦 Total especies: {df_counts.shape[0]}")
print(f"📦 Total embeddings: {df_counts['num_embeddings'].sum()}")

df_counts.head(10)

🐦 Total especies: 104
📦 Total embeddings: 92271


,species,num_embeddings
52,dendrocopos major,900
1,calidris alpina,900
36,phoenicurus ochruros,900
82,turdus iliacus,900
38,motacilla flava,900
81,turdus viscivorus,900
41,lullula arborea,900
79,dryocopus martius,900
78,tachybaptus ruficollis,900
44,gallinago gallinago,900


In [ ]:
Entreno con Modelo 

In [25]:
import os
import numpy as np
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, top_k_accuracy_score

# =====================================
# CONFIGURACIÓN
# =====================================
EMB_DIR = "embeddings_perch"
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_ITER = 500
C_VALUE = 1.0
TOP_K = 3

# =====================================
# CARGAR EMBEDDINGS + POOLING TEMPORAL
# =====================================
X = []
y = []

print("📥 Cargando embeddings (con average pooling temporal)...")

for species in tqdm(os.listdir(EMB_DIR)):
    species_dir = os.path.join(EMB_DIR, species)
    if not os.path.isdir(species_dir):
        continue

    for fname in os.listdir(species_dir):
        if not fname.endswith(".npz"):
            continue

        path = os.path.join(species_dir, fname)
        data = np.load(path)

        emb = data["embedding"]

        # 🔥 FIX CLAVE: convertir embedding temporal -> vector fijo
        # Perch suele dar (T, D)
        if emb.ndim == 2:
            emb = emb.mean(axis=0)

        X.append(emb)
        y.append(species)

X = np.array(X)
y = np.array(y)

print(f"✅ Embeddings cargados: {X.shape[0]}")
print(f"📐 Dimensión final embedding: {X.shape[1]}")
print(f"🐦 Número especies: {len(np.unique(y))}")

# =====================================
# CODIFICAR ETIQUETAS
# =====================================
le = LabelEncoder()
y_enc = le.fit_transform(y)

# =====================================
# SPLIT ESTRATIFICADO
# =====================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_enc,
    test_size=TEST_SIZE,
    stratify=y_enc,
    random_state=RANDOM_STATE
)

print(f"📊 Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

# =====================================
# PIPELINE: NORMALIZACIÓN + SOFTMAX
# =====================================
pipeline = Pipeline([
    ("scaler", StandardScaler()),   # normalización por feature
    ("clf", LogisticRegression(
        multi_class="multinomial",  # softmax real
        solver="lbfgs",
        C=C_VALUE,
        max_iter=MAX_ITER,
        n_jobs=-1
    ))
])

# =====================================
# ENTRENAR
# =====================================
print("🚀 Entrenando regresión logística multinomial...")
pipeline.fit(X_train, y_train)
print("✅ Entrenamiento completado")

# =====================================
# EVALUACIÓN
# =====================================
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
top3 = top_k_accuracy_score(y_test, y_proba, k=TOP_K)

print("\n📈 RESULTADOS")
print(f"🎯 Top-1 Accuracy: {acc:.4f}")
print(f"🥉 Top-{TOP_K} Accuracy: {top3:.4f}")

# =====================================
# GUARDAR MODELO
# =====================================
import joblib

joblib.dump(pipeline, "perch_logreg_softmax.joblib")
joblib.dump(le, "label_encoder.joblib")

print("💾 Modelo y label encoder guardados")


📥 Cargando embeddings (con average pooling temporal)...


100%|██████████| 104/104 [00:29<00:00,  3.51it/s]


✅ Embeddings cargados: 92271
📐 Dimensión final embedding: 1280
🐦 Número especies: 104
📊 Train: 73816 | Test: 18455
🚀 Entrenando regresión logística multinomial...


/opt/anaconda3/envs/perch_audio/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


✅ Entrenamiento completado

📈 RESULTADOS
🎯 Top-1 Accuracy: 0.7682
🥉 Top-3 Accuracy: 0.8543
💾 Modelo y label encoder guardados


In [ ]:
Probar el modelo con codigo para mac

In [4]:
import audioclass
print(dir(audioclass))

['BaseIterator', 'BatchGenerator', 'ClipClassificationModel', 'ModelOutput', 'SimpleIterator', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'batch', 'constants', 'load_clip', 'load_recording', 'models', 'postprocess', 'preprocess', 'process_iterable', 'utils']


In [1]:
import sys
# 1. Limpiar rastro de versiones fallidas en memoria
if 'tensorflow' in sys.modules:
    del sys.modules['tensorflow']

# 2. Importar de nuevo con un pequeño truco de recarga
import tensorflow as tf
try:
    print(f"Versión detectada: {tf.__version__}")
except AttributeError:
    print("Reintentando carga profunda...")
    import importlib
    importlib.reload(tf)
    print(f"Versión corregida: {tf.__version__}")

# 3. Ahora sí, el resto de librerías
import tensorflow_hub as hub
import librosa
import joblib
import gradio as gr

print("✅ ¡Todo cargado! Ya puedes usar el modelo PERCH.")

Versión detectada: 2.15.0


/opt/anaconda3/envs/bio_m2/lib/python3.10/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/opt/anaconda3/envs/bio_m2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ ¡Todo cargado! Ya puedes usar el modelo PERCH.


In [2]:
import warnings
warnings.filterwarnings('ignore') # Limpia la pantalla de avisos innecesarios

# --- CONFIGURACIÓN Y LANZAMIENTO ---
# Asegúrate de que estos nombres coinciden con tus archivos grabados
PIPELINE_FILE = "perch_logreg_softmax.joblib"
LABEL_ENCODER_FILE = "label_encoder.joblib"

try:
    # Instanciamos el predictor usando la clase que definimos antes
    # (Si no la tienes en esta celda, asegúrate de haber corrido la celda donde está definida)
    predictor = BirdClassifierM2(PIPELINE_FILE, LABEL_ENCODER_FILE)

    # Creamos la interfaz de Gradio
    app = gr.Interface(
        fn=predictor.predict,
        inputs=gr.Audio(type="filepath", label="Cargar audio de Xeno-Canto (MP3/WAV)"),
        outputs=gr.Textbox(label="Identificación de Ave"),
        title="🐦 BirdID - Motor PERCH v4 (M2)",
        description="Identificación bioacústica inmediata usando Apple Silicon.",
        theme=gr.themes.Soft()
    )

    # Lanzamos la interfaz
    app.launch(inline=True, share=False)
    
except Exception as e:
    print(f"❌ Error al iniciar la interfaz: {e}")

❌ Error al iniciar la interfaz: name 'BirdClassifierM2' is not defined


In [5]:
# Probando el modelo codigo ajustado para mac

import os
# Bloqueo total de GPU para estabilidad en Apple Silicon M2
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import librosa
import joblib
import gradio as gr
import time
import warnings
import tensorflow as tf
import tensorflow_hub as hub

warnings.filterwarnings('ignore')

# --- CONFIGURACIÓN DE RUTAS ---
CARPETA_MODELO = "perch_model" 
PIPELINE_FILE = "perch_logreg_softmax.joblib"
LABEL_ENCODER_FILE = "label_encoder.joblib"

class BirdClassifierM2:
    def __init__(self, pipeline_path, le_path, model_path):
        print(f"🚀 Iniciando motor PERCH Pro (1280 dim) en CPU...")
        with tf.device('/CPU:0'):
            if os.path.exists(model_path):
                self.perch_model = hub.load(model_path)
                print("✅ Modelo PERCH cargado.")
            else:
                raise FileNotFoundError(f"❌ No se encontró la carpeta: {model_path}")
        
        self.pipeline = joblib.load(pipeline_path)
        self.label_encoder = joblib.load(le_path)
        print("✅ Clasificadores Softmax y Label Encoder listos.")

    def process_audio_multi_window(self, audio_path):
        # 1. Cargar audio completo a 32kHz
        audio, _ = librosa.load(audio_path, sr=32000, mono=True)
        
        # 2. Definir tamaño de ventana (5 segundos = 160,000 muestras)
        window_size = 160000 
        audio_len = len(audio)
        
        # Si el audio es más corto que 5 segundos, rellenamos
        if audio_len < window_size:
            audio = np.pad(audio, (0, window_size - audio_len), mode='constant')
            audio_chunks = [audio]
        else:
            # Dividir en ventanas de 5 segundos sin solapamiento
            audio_chunks = [audio[i:i + window_size] for i in range(0, audio_len - window_size + 1, window_size)]

        all_embeddings = []
        
        with tf.device('/CPU:0'):
            for chunk in audio_chunks:
                # Convertir cada trozo a tensor (1, 160000)
                tensor = tf.convert_to_tensor(chunk[np.newaxis, :], dtype=tf.float32)
                outputs = self.perch_model.infer_tf(tensor)
                
                # Extraer embedding (1280 dimensiones)
                if isinstance(outputs, dict):
                    emb = outputs['embedded_feature']
                else:
                    emb = outputs[1] if isinstance(outputs, (list, tuple)) else outputs
                
                all_embeddings.append(np.mean(emb, axis=0))

        # Promediar todos los trozos para obtener un vector robusto de (1, 1280)
        final_embedding = np.mean(all_embeddings, axis=0).reshape(1, -1)
        return final_embedding

    def predict(self, audio_path):
        if audio_path is None: return "Por favor, sube un archivo de audio."
        try:
            start_t = time.time()
            
            # Obtener embedding robusto promediado
            features = self.process_audio_multi_window(audio_path)
            
            # Predicción y Probabilidades
            probs = self.pipeline.predict_proba(features)
            confianza = np.max(probs)
            
            # --- MEJORA: UMBRAL DE SEGURIDAD (40%) ---
            if confianza < 0.40:
                return (f"❓ Especie no identificada con claridad\n"
                        f"📊 Confianza insuficiente: {confianza:.2%}\n"
                        f"📝 Nota: El sonido es muy débil o la especie no está en el entrenamiento.")
            
            pred_numeric = self.pipeline.predict(features)
            especie_nombre = self.label_encoder.inverse_transform(pred_numeric)
            
            duration = time.time() - start_t
            nombre = especie_nombre[0] if isinstance(especie_nombre, (list, np.ndarray)) else especie_nombre
            
            return (f"🐦 Especie: {nombre}\n"
                    f"📊 Confianza: {confianza:.2%}\n"
                    f"⚡ Tiempo M2 (Multi-ventana): {duration:.2f}s")
        
        except Exception as e:
            return f"❌ Error en el análisis: {str(e)}"

# --- LANZAMIENTO ---
try:
    predictor = BirdClassifierM2(PIPELINE_FILE, LABEL_ENCODER_FILE, CARPETA_MODELO)

    app = gr.Interface(
        fn=predictor.predict,
        inputs=gr.Audio(type="filepath", label="Cargar Canto (Cualquier duración)"),
        outputs=gr.Textbox(label="Identificación Inteligente"),
        title="BirdID Ecuador Pro - Apple Silicon M2",
        description="Analiza el audio completo en ventanas de 5s para máxima precisión (PERCH 1280dim).",
        theme=gr.themes.Soft()
    )

    print("✅ Iniciando interfaz con análisis multi-ventana...")
    app.launch(inline=True, share=False)
    
except Exception as e:
    print(f"❌ Error al iniciar: {e}")

🚀 Iniciando motor PERCH Pro (1280 dim) en CPU...
✅ Modelo PERCH cargado.
✅ Clasificadores Softmax y Label Encoder listos.
✅ Iniciando interfaz con análisis multi-ventana...
* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
